In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [3]:
df = pd.read_csv(r'/home/user/Downloads/DataHut_AT_Billa_PriceExtractions_20260720.CSV', low_memory=False, sep='|')

In [4]:
df.shape

(12386, 21)

In [5]:
empty_cols = df.columns[df.isna().all()].tolist()
empty_cols

['price_valid_from',
 'percentage_discount',
 'promotion_valid_from',
 'promotion_valid_upto',
 'region',
 'pack_size']

In [5]:
req_cols = """unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom"""

req_cols = [c.strip() for c in req_cols.split(",") if c.strip()]

missing = [c for c in req_cols if c not in df.columns]
extra = [c for c in df.columns if c not in req_cols]

print("Requirement columns:", len(req_cols))
print("File columns:", len(df.columns))
print("Missing:", missing)
print("Extra:", extra)
print("Order matches:", req_cols == list(df.columns))

Requirement columns: 21
File columns: 21
Missing: []
Extra: []
Order matches: True


In [8]:
# Empty columns as per requirement
req_empty = """
store_name,store_addressline1,store_addressline2,store_suburb,store_state,store_postcode,store_addressid,brand_type,drained_weight,producthierarchy_level7,promotion_valid_from,promotion_valid_upto,promotion_type,percentage_discount,package_sizeof_sellingprice,per_unit_sizedescription,price_valid_from,multi_buy_item_count,multi_buy_items_price_total,variants,instructions,age_of_the_product,age_recommendations,flavour,nutritions,vitamins,labelling,grade,region,packaging,receipies,processed_food,barcode,frozen,chilled,cooking_part,handmade,max_heating_temperature,special_information,label_information,dimensions,special_nutrition_purpose,feeding_recommendation,warranty,color,model_number,material,usp,dosage_recommendation,food_preservation,size,rating,review,file_name_1,file_name_2,file_name_3,file_name_4,file_name_5,file_name_6,competitor_product_key,fit_guide,occasion,material_composition,style,care_instructions,heel_type,heel_height,upc,dietary_lifestyle,manufacturer_address,importer_address,distributor_address,vinification_details,recycling_information,return_address,beer_deg,netcontent,netweight,random_weight_flag,instock,promo_limit,multibuy_items_pricesingle,perfect_match,servings_per_pack,warning,suitable_for,standard_drinks,environmental,retail_limit"""

req_empty = [c.strip() for c in req_empty.split(",") if c.strip()]

# Empty columns in the file
file_empty = df.columns[df.isna().all()].tolist()

# Compare
missing_empty = [c for c in req_empty if c not in file_empty]
unexpected_empty = [c for c in file_empty if c not in req_empty]
matching_empty = [c for c in req_empty if c in file_empty]

print("Requirement empty columns :", len(req_empty))
print("File empty columns        :", len(file_empty))
print("Matching                 :", len(matching_empty))
print("Expected empty but not empty:", missing_empty)
print("Unexpected empty columns     :", unexpected_empty)

Requirement empty columns : 89
File empty columns        : 88
Matching                 : 88
Expected empty but not empty: ['age_of_the_product']
Unexpected empty columns     : []


In [9]:
df.instock.value_counts()

Series([], Name: count, dtype: int64)

In [6]:
df.columns

Index(['unique_id', 'competitor_name', 'extraction_date', 'product_name',
       'grammage_quantity', 'grammage_unit', 'regular_price', 'selling_price',
       'price_valid_from', 'price_per_unit', 'percentage_discount',
       'promotion_price', 'promotion_valid_from', 'promotion_valid_upto',
       'promotion_description', 'currency', 'breadcrumb', 'pdp_url', 'region',
       'pack_size', 'site_shown_uom'],
      dtype='object')

In [8]:
df.unique_id.nunique()

12386

In [9]:
df.competitor_name.unique()

array(['billa'], dtype=object)

In [10]:
df.currency.value_counts()

currency
EUR    12386
Name: count, dtype: int64

In [11]:
df.extraction_date.value_counts()

extraction_date
2026-07-20    12386
Name: count, dtype: int64

In [12]:
df[df.selling_price==0]

,unique_id,competitor_name,extraction_date,product_name,grammage_quantity,grammage_unit,regular_price,selling_price,price_valid_from,price_per_unit,percentage_discount,promotion_price,promotion_valid_from,promotion_valid_upto,promotion_description,currency,breadcrumb,pdp_url,region,pack_size,site_shown_uom


In [13]:
pattern = r'^[^>]+( > [^>]+)*$'

# Invalid breadcrumbs
invalid = df[
    df["breadcrumb"].isna() |
    (df["breadcrumb"].str.strip() == "") |
    (~df["breadcrumb"].str.fullmatch(pattern, na=False))
]

print(f"Total invalid breadcrumbs: {len(invalid)}")

if not invalid.empty:
    print(invalid[["breadcrumb"]])

Total invalid breadcrumbs: 0


In [14]:
df.columns.to_list()

['unique_id',
 'competitor_name',
 'extraction_date',
 'product_name',
 'grammage_quantity',
 'grammage_unit',
 'regular_price',
 'selling_price',
 'price_valid_from',
 'price_per_unit',
 'percentage_discount',
 'promotion_price',
 'promotion_valid_from',
 'promotion_valid_upto',
 'promotion_description',
 'currency',
 'breadcrumb',
 'pdp_url',
 'region',
 'pack_size',
 'site_shown_uom']

In [15]:
import pandas as pd

price_columns = [
    "regular_price",
    "selling_price",
    "price_was",
    "promotion_price",
    "percentage_discount",
    "price_per_unit",
    "multi_buy_items_price_total",
    "multibuy_items_pricesingle"
]

for col in price_columns:
    if col not in df.columns:
        continue

    s = df[col].fillna("").astype(str)

    issues = {
        "Zero values": pd.to_numeric(s, errors="coerce") == 0,
        "Negative values": pd.to_numeric(s, errors="coerce") < 0,
        "Multiple decimal points": s.str.count(r"\.") > 1,
        "More than 2 decimal places": s.str.contains(r"\.\d{3,}$", regex=True),
        "Non-numeric values": (~s.str.match(r"^-?\d+(\.\d+)?$", na=False)) & (s != ""),
        "Leading/Trailing whitespace": s != s.str.strip(),
    }

    print(f"\n{'='*15} {col} {'='*15}")

    for issue, mask in issues.items():
        if mask.any():
            print(f"{issue}: {mask.sum()}")
            print(df.loc[mask, [col]].head(5).to_string(index=False))


=============== regular_price ===============

=============== selling_price ===============

=============== promotion_price ===============

=============== percentage_discount ===============

=============== price_per_unit ===============
Multiple decimal points: 328
     price_per_unit
1 kg Abtr.G 22.92 €
1 kg Abtr.G 12.63 €
 1 kg Abtr.G 8.45 €
1 kg Abtr.G 18.60 €
 1 kg Abtr.G 7.83 €
Non-numeric values: 11016
     price_per_unit
1 kg Abtr.G 22.92 €
       100 g 2.39 €
       100 g 2.55 €
       1 Stk 1.06 €
       1 kg 17.48 €


In [16]:
import re

col = "selling_price"

# More than one decimal point in the numeric part
invalid = df[
    df[col]
    .fillna("")
    .astype(str)
    .str.contains(r'\d+\.\d+\.', regex=True)
]

print(f"Rows with multiple decimal points: {len(invalid)}")
print(invalid[[col]])

Rows with multiple decimal points: 0
Empty DataFrame
Columns: [selling_price]
Index: []


In [17]:
import re

url_pattern = re.compile(
    r"^https?://[^\s/$.?#].[^\s]*$",
    re.IGNORECASE
)

invalid_urls = df[
    df["pdp_url"].isna() |
    ~df["pdp_url"].astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_urls)}")
print(invalid_urls[["unique_id", "pdp_url"]])

Invalid URLs: 0
Empty DataFrame
Columns: [unique_id, pdp_url]
Index: []


In [18]:
import re

image_cols = [c for c in df.columns if c.startswith("image_url_")]

# Basic image URL pattern
pattern = re.compile(
    r"^https?://[^\s]+\.(jpg|jpeg|png|webp|gif|bmp|avif)(\?.*)?$",
    re.IGNORECASE
)

for col in image_cols:
    invalid = df[
        df[col].notna() &
        (df[col].astype(str).str.strip() != "") &
        ~df[col].astype(str).str.match(pattern)
    ]

    print(f"\n{col}: {len(invalid)} invalid URLs")

    if not invalid.empty:
        print(invalid[["unique_id", col]].head())

In [19]:
import re

pattern = re.compile(r"^https?://[^\s]+$", re.IGNORECASE)

for col in image_cols:
    invalid = df[
        df[col].notna() &
        (df[col].astype(str).str.strip() != "") &
        ~df[col].astype(str).str.match(pattern)
    ]

    print(f"{col}: {len(invalid)} invalid URLs")

In [17]:
price_cols = [
    "regular_price",
    "selling_price",
   
    "price_per_unit"
   
]

# Keep only columns that exist in the file
price_cols = [c for c in price_cols if c in df.columns]

# Rows where at least one price column is empty
empty_price_rows = df[df[price_cols].replace("", pd.NA).isna().any(axis=1)]

print(f"Rows with empty price fields: {len(empty_price_rows)}")
display(empty_price_rows[["unique_id", "product_name"] + price_cols])

Rows with empty price fields: 16572


,unique_id,product_name,regular_price,selling_price,price_per_unit
6,2020006204071,DESPAR Lemon,1.99,1.99,NaN
7,2020001001880,DESPAR Vino Rosso,0.79,0.79,NaN
8,2020006204156,DESPAR Peach,1.99,1.99,NaN
9,2020006203982,DESPAR Ice Tea Lemon-Lime,1.99,1.99,NaN
10,2020001001859,DESPAR Vino Bianco,0.79,0.79,NaN
27,7375477,SPAR Natur*pur Bio-Entspannungs- Kräutertee 20 Teebeutel,2.99,2.99,NaN
30,2020000523789,SPAR office Lineal 30cm,1.29,1.29,NaN
33,7939464,SPAR office nature Kuverts C5 ohne Fenster 20 Stück,1.69,1.69,NaN
34,2020002398880,SPAR Schaschlik Spieße 20 cm 50 Stück,0.99,0.99,NaN
35,2020004093172,SPAR Muffinförmchen Stern 60 Stück,2.99,2.99,NaN


In [16]:
import pandas as pd

# Treat empty strings as missing values
promo_price = df["promotion_price"].replace("", pd.NA)
promo_desc = df["promotion_description"].replace("", pd.NA)

# 1. Promotion price present but promotion description missing
price_no_desc = df[
    promo_price.notna() & promo_desc.isna()
]

# 2. Promotion description present but promotion price missing
desc_no_price = df[
    promo_desc.notna() & promo_price.isna()
]

print("Promotion price present but description missing:", len(price_no_desc))
display(price_no_desc[["unique_id", "product_name", "promotion_price", "promotion_description"]])

print("\nPromotion description present but price missing:", len(desc_no_price))
display(desc_no_price[["unique_id", "product_name", "promotion_price", "promotion_description"]])

Promotion price present but description missing: 0


,unique_id,product_name,promotion_price,promotion_description



Promotion description present but price missing: 0


,unique_id,product_name,promotion_price,promotion_description


In [23]:
invalid = df[df["breadcrumb"].str.contains("…", na=False)]

print(f"Rows with truncated breadcrumbs: {len(invalid)}")
display(invalid[["unique_id", "breadcrumb"]])

Rows with truncated breadcrumbs: 0


,unique_id,breadcrumb


In [20]:
import pandas as pd

# Text columns only
text_cols = df.select_dtypes(include="object").columns

invalid_chars = [ "\n", "\r", "\t"]

issues = {}

for col in text_cols:
    mask = df[col].fillna("").astype(str).str.contains(r"[\n\r\t]", regex=True)
    if mask.any():
        issues[col] = df.loc[mask, ["unique_id", col]]

# Summary
if issues:
    print("Columns containing \\n, \\r or \\t:\n")
    for col, data in issues.items():
        print(f"{col}: {len(data)} rows")
else:
    print("No \\n, \\r or \\t found.")
    

No \n, \r or \t found.


In [21]:
issues

{}

In [22]:
columns = ["product_description", "special_information", "manufacturer_address"]

for col in columns:
    rows = df[
        df[col]
        .fillna("")
        .astype(str)
        .str.contains("|", regex=False, na=False)
    ]

    print(f"\n=== {col}: {len(rows)} rows ===")

    if not rows.empty:
        display(rows[["unique_id", "product_name", col]])

KeyError: 'product_description'

In [27]:
import pandas as pd
import re




df["dimensions"] = df["dimensions"].fillna("").str.strip()

pattern = re.compile(
    r'^\d+(\.\d+)?(\s*[a-zA-Z]+)?\s*[xX×]\s*'
    r'\d+(\.\d+)?(\s*[a-zA-Z]+)?\s*[xX×]\s*'
    r'\d+(\.\d+)?(\s*[a-zA-Z]+)?$'
)

invalid = df[
    (df["dimensions"] != "") &
    (~df["dimensions"].str.match(pattern))
]

print(f"Invalid values: {len(invalid)}")
print(invalid[["unique_id", "product_name", "dimensions"]])

KeyError: 'dimensions'

In [23]:

# Replace empty strings with NA
df = df.replace(r'^\s*$', pd.NA, regex=True)

# Filter Fleisch & Fisch
ff = df[df["producthierarchy_level1"] == "Fleisch & Fisch"]

# Subcategories that should have price_per_unit
ppu_subcats = ["Fleisch & Geflügel", "Frischfisch & Schalentiere"]

# Missing price_per_unit
invalid_ppu = ff[
    ff["producthierarchy_level2"].isin(ppu_subcats) &
    ff["price_per_unit"].isna()
]

print("Missing price_per_unit:")
print(invalid_ppu[["unique_id","product_name","producthierarchy_level2","price_per_unit"]])

KeyError: 'producthierarchy_level1'

In [24]:
date_columns = [
    "promotion_valid_upto",
    "promotion_valid_from",
    "price_valid_from"
]

for col in date_columns:
    print("\nColumn:", col)
    print(df[col].value_counts(dropna=False))


Column: promotion_valid_upto
promotion_valid_upto
NaN    12386
Name: count, dtype: int64

Column: promotion_valid_from
promotion_valid_from
NaN    12386
Name: count, dtype: int64

Column: price_valid_from
price_valid_from
NaN    12386
Name: count, dtype: int64


In [25]:
invalid_discount = df[
    df["percentage_discount"]
    .astype(str)
    .str.contains(r"[%\-]", regex=True, na=False)
]

print("Invalid percentage_discount values:")
print(invalid_discount[["unique_id", "product_name", "percentage_discount"]])

print("\nCount:", len(invalid_discount))

Invalid percentage_discount values:
Empty DataFrame
Columns: [unique_id, product_name, percentage_discount]
Index: []

Count: 0


In [26]:
invalid = df[
    df["promotion_description"].notna() &
    (
        df["promotion_valid_from"].isna() |
        df["promotion_valid_upto"].isna()
    )
]

print(f"Rows with promotion description but missing promotion dates: {len(invalid)}")

display(
    invalid[
        [
            "unique_id",
            "product_name",
            "promotion_description",
            "promotion_valid_from",
            "promotion_valid_upto",
            "pdp_url"
        ]
    ]
)

Rows with promotion description but missing promotion dates: 895


,unique_id,product_name,promotion_description,promotion_valid_from,promotion_valid_upto,pdp_url
4,00-10042,Iglo Pazifischer Polar-Dorsch paniert 400 g Packung,Bei 3 Packungen je,NaN,NaN,https://shop.billa.at/produkte/iglo-pazifischer-polardorsch-paniert-0010042
57,00-13992,Iglo Polar-Dorsch Knusperhülle 300 g Packung,Bei 3 Packungen je,NaN,NaN,https://shop.billa.at/produkte/iglo-polardorsch-knusperhuelle-0013992
64,00-10351,Rauch Happy Day Apfelsaft 1 liter Packung Einweg,Bei 3 Packungen je,NaN,NaN,https://shop.billa.at/produkte/rauch-happy-day-apfelsaft-0010351
82,00-10827,Rauch Happy Day Multivitaminsaft 1 liter Packung Einweg,Bei 3 Packungen je,NaN,NaN,https://shop.billa.at/produkte/rauch-happy-day-multivitaminsaft-0010827
90,00-11168,Stolichnaya Vodka 0.7 liter Flasche,AKTION,NaN,NaN,https://shop.billa.at/produkte/stolichnaya-vodka-0011168
135,00-16395,Knorr Meisterkessel Südtiroler Bauernsuppe 500 g Dose,"ab 2 Dosen, AKTION",NaN,NaN,https://shop.billa.at/produkte/knorr-meisterkessel-suedtiroler-bauernsuppe-0016395
156,00-14294,Mutti Tomaten Polpa 400 g Dose,"ab 2 Dosen, AKTION",NaN,NaN,https://shop.billa.at/produkte/mutti-tomaten-polpa-0014294
242,00-17973,Leibniz Zoo 125 g Packung,Bei 3 Packungen je,NaN,NaN,https://shop.billa.at/produkte/leibniz-zoo-0017973
280,00-200001,Gourmet Gold Häppchen Kalb & Gemüse 85 g Dose,Bei 5 Dosen je,NaN,NaN,https://shop.billa.at/produkte/gourmet-gold-haeppchen-kalb-und-gemuese-00200001
281,00-19893,Delverde Spaghetti 1 kg Packung,AKTION,NaN,NaN,https://shop.billa.at/produkte/delverde-spaghetti-0019893


In [7]:
import pandas as pd

# Normalize for case-insensitive comparison
df["site_shown_uom"] = df["site_shown_uom"].fillna("").astype(str)
df["grammage_unit"] = df["grammage_unit"].fillna("").astype(str)
df["grammage_quantity"] = df["grammage_quantity"].fillna("").astype(str)

# -----------------------------
# Rule 1: Teebeutel / Beutel -> btl
# -----------------------------
btl_keywords = [
    "Teebeutel Packung",
    "Teebeutel",
    "Beutel",
    "Btl",
    "Teebeutel Karton",
    "Teebeutel Paket",
    "Packung Beutel"
]

btl_invalid = df[
    df["site_shown_uom"].str.contains("|".join(btl_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "btl")
]

print(f"BTL unit mismatches: {len(btl_invalid)}")
display(btl_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit"
]])

# -----------------------------
# Rule 2: Portion Packung / ANW -> stuck
# -----------------------------
stuck_keywords = [
    "Portion Packung",
    "ANW"
]

stuck_invalid = df[
    df["site_shown_uom"].str.contains("|".join(stuck_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "stuck")
]

print(f"STUCK unit mismatches: {len(stuck_invalid)}")
display(stuck_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit"
]])

# -----------------------------
# Rule 3: wg / Waschgänge -> wg
# -----------------------------
wg_keywords = [
    "wg",
    "Waschgänge"
]

wg_invalid = df[
    df["site_shown_uom"].str.contains("|".join(wg_keywords), case=False, regex=True)
    &
    (df["grammage_unit"].str.lower() != "wg")
]

print(f"WG unit mismatches: {len(wg_invalid)}")
display(wg_invalid[[
    "unique_id",
    "product_name",
    "site_shown_uom",
    "grammage_quantity",
    "grammage_unit",
    "pdp_url"
]])

BTL unit mismatches: 0


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit


STUCK unit mismatches: 0


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit


WG unit mismatches: 0


,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,pdp_url


In [27]:
import re
import pandas as pd

# Normalize grammage_unit
df["grammage_unit_norm"] = df["grammage_unit"].fillna("").str.lower().str.strip()

# Extract quantity from site_shown_uom
df["site_qty"] = (
    df["site_shown_uom"]
      .str.extract(r"(\d+(?:[.,]\d+)?)", expand=False)
      .str.replace(",", ".", regex=False)
)

df["site_qty"] = pd.to_numeric(df["site_qty"], errors="coerce")

# ---------- Rule 1 : Beutel / Btl ----------
btl_pattern = (
    r"Teebeutel Packung|Teebeutel Karton|Teebeutel Paket|"
    r"Packung Beutel|Teebeutel|Beutel|Btl"
)

btl_issue = df[
    df["site_shown_uom"].fillna("").str.contains(btl_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "btl") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

# ---------- Rule 2 : Portion Packung / ANW ----------
stuck_pattern = r"Portion Packung|ANW"

stuck_issue = df[
    df["site_shown_uom"].fillna("").str.contains(stuck_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "stuck") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

# ---------- Rule 3 : Waschgang / Waschgänge / wg ----------
wg_pattern = r"\bwg\b|Waschgänge|Waschgang"

wg_issue = df[
    df["site_shown_uom"].fillna("").str.contains(wg_pattern, case=False, regex=True)
    &
    (
        (df["grammage_unit_norm"] != "wg") |
        (df["grammage_quantity"] != df["site_qty"])
    )
]

print("Btl issues:", len(btl_issue))
print("Stuck issues:", len(stuck_issue))
print("WG issues:", len(wg_issue))

Btl issues: 1181
Stuck issues: 0
WG issues: 86


In [28]:
btl_issue[
    [
        "unique_id",
        "product_name",
        "site_shown_uom",
        "grammage_quantity",
        "grammage_unit",
        "site_qty"
    ]
].head(20)

,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit,site_qty
11,00-101511,Toppits Gefrierbeutel 1L 40 Beutel Packung,40 Beutel Packung,1,stück,40.0
14,00-101513,Toppits Gefrierbeutel 6 Liter 20 Beutel Packung,20 Beutel Packung,1,stück,20.0
35,00-129148,Teekanne Fixbutte 40 Teebeutel Paket,40 Teebeutel Paket,40,btl,40.0
39,00-129163,Teekanne Fixminze 40 Teebeutel Paket,40 Teebeutel Paket,40,btl,40.0
40,00-129189,Teekanne Fixmille 40 Teebeutel Packung,40 Teebeutel Packung,40,btl,40.0
47,00-133710,Twinings Earl Grey 50 Teebeutel Packung,50 Teebeutel Packung,50,btl,50.0
48,00-133728,Twinings English Breakfast 50 Teebeutel Packung,50 Teebeutel Packung,50,btl,50.0
65,00-103992,Vegeta Würzmischung 1 kg Beutel,1 kg Beutel,1,kg,1.0
91,00-11080,Knorr Basis für Faschiertes 1 Packung Beutel,1 Packung Beutel,1,btl,1.0
106,00-116129,Kinder Schokobons 200 g Beutel,200 g Beutel,200,g,200.0


In [29]:
wrong_unit_wg = wg_issue[
    wg_issue["grammage_unit_norm"] != "wg"
]

wrong_qty_wg = wg_issue[
    (wg_issue["grammage_unit_norm"] == "wg") &
    (wg_issue["grammage_quantity"] != wg_issue["site_qty"])
]

print("WG - Wrong unit:", len(wrong_unit_wg))
print("WG - Wrong quantity:", len(wrong_qty_wg))

WG - Wrong unit: 82
WG - Wrong quantity: 4


In [30]:
wrong_unit_wg[
    [
        "unique_id",
        "product_name",
        "site_shown_uom",
        "grammage_quantity",
        "grammage_unit"
    ]
]

,unique_id,product_name,site_shown_uom,grammage_quantity,grammage_unit
1631,00-352456,Clever Vollwaschmittel Flüssig 40 Waschgang Flasche,40 Waschgang Flasche,1,stück
1634,00-352454,Clever Colorwaschmittel Flüssig 40 Waschgang Flasche,40 Waschgang Flasche,1,stück
2036,00-393820,Persil Gel Color Activ 100 Waschgang Flasche,100 Waschgang Flasche,1,stück
3449,00-451972,bi good Apfelblüte Colorwaschmittel Beutel 25 Waschgang Beutel,25 Waschgang Beutel,1,stück
3450,00-451977,bi good Lavendel Vollwaschmittel Beutel 25 Waschgang Beutel,25 Waschgang Beutel,1,stück
3717,00-471467,Lenor Weichspüler Goldene Orchidee 38 Waschgang Flasche,38 Waschgang Flasche,1,stück
3820,00-465128,Sagrotan Hygienespüler 20 Waschgang Flasche,20 Waschgang Flasche,1,stück
4088,00-482548,Weißer Riese Trio Caps Lotus 40 Waschgang Karton,40 Waschgang Karton,1,stück
4089,00-482549,Weißer Riese Trio Caps Orchidee 40 Waschgang Karton,40 Waschgang Karton,1,stück
4756,00-572851,Dr. Beckmann Magic Leaves Universal 25 Waschgang Stück,25 Waschgang Stück,1,stück
